# WSJ XXXX-XXXX → Date + Headline + Filename (full corpus)

Minimal 3-column export over every XML in the WSJ folder. No keyword filtering, no body text.

Schema (per sample 134001711.xml, confirmed in `proquest_tdm_export.ipynb`):
- Date: `<NumericDate>YYYY-MM-DD</NumericDate>` (backup `<StartDate>`)
- Headline: `<TitleAtt><Title>...</Title></TitleAtt>`
- Filename: `f.name` (e.g. `134001711.xml`)

Six cells:
1. **Scan** all XML files with regex on raw bytes → DataFrame → parquet (snappy) + csv.gz
2. **Peek** at the snappy file (file sizes, head/tail, random sample, sanity checks)
3. **Shrink** — re-write as zstd-22 + drop the uniform `.xml` suffix (usually clears the 30 MB cap)
4. **Peek** at the shrunk file (file size, sample rows, sanity checks)
5. **Split** (fallback) — only if Cell 4 shows ⚠; chunk by year range into ≤ 28 MB parquets
6. **Export** — `aws s3 cp` to the results bucket. The TDM cap is weekly, so ship one chunk per week if you went the split route.

Recover the original filename downstream as `f'{article_id}.xml'`.

In [1]:
#conda install pyarrow fastparquet

In [2]:
# ===== Cell 1: Extract date + headline + filename from every WSJ XML =====
import re, time, html
from pathlib import Path
import pandas as pd

candidates = [Path('data/WSJ_1920-1964'), Path('/data/WSJ_1920-1964'),
              Path('../data/WSJ_1920-1964'), Path('/home/jovyan/data/WSJ_1920-1964')] #Change this to the appropriate time
WSJ_ROOT = next((p for p in candidates if p.exists()), None)
if WSJ_ROOT is None:
    raise SystemExit('No WSJ folder found at any candidate path.')
print(f'WSJ_ROOT = {WSJ_ROOT}')

OUT_DIR = Path('../ProQuest TDM Studio Samples/output_files/')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PARQUET = OUT_DIR / 'wsj_headlines_1920_1964.parquet'
OUT_CSVGZ   = OUT_DIR / 'wsj_headlines_1920_1964.csv.gz'

DATE_RE        = re.compile(rb'<NumericDate>\s*(\d{4}-\d{2}-\d{2})', re.I)
DATE_RE_BACKUP = re.compile(rb'<StartDate>\s*(\d{4}-?\d{2}-?\d{2})', re.I)
# <TitleAtt><Title>...</Title></TitleAtt> — first <Title> after <TitleAtt>; .*? non-greedy across newlines
TITLE_RE       = re.compile(rb'<TitleAtt>.*?<Title>(.*?)</Title>', re.I | re.S)
TITLE_RE_BACKUP = re.compile(rb'<Title>(.*?)</Title>', re.I | re.S)
WS_RE          = re.compile(r'\s+')

def extract_date(buf):
    m = DATE_RE.search(buf) or DATE_RE_BACKUP.search(buf)
    return m.group(1).decode() if m else None

def extract_title(buf):
    m = TITLE_RE.search(buf) or TITLE_RE_BACKUP.search(buf)
    if not m:
        return ''
    raw = m.group(1).decode('utf-8', errors='replace')
    # Title text may contain HTML entities (&amp;, &lt;, etc.); decode and collapse whitespace.
    return WS_RE.sub(' ', html.unescape(raw)).strip()

files = list(WSJ_ROOT.glob('*.xml'))
print(f'Total XML files: {len(files):,}')

rows = []
no_date = no_title = 0
t0 = time.time()
for i, f in enumerate(files):
    try:
        buf = f.read_bytes()
    except Exception:
        continue
    d = extract_date(buf)
    t = extract_title(buf)
    if d is None: no_date += 1
    if not t:    no_title += 1
    rows.append({'date': d, 'headline': t, 'fname': f.name})
    if (i + 1) % 10000 == 0:
        rate = (i + 1) / (time.time() - t0)
        eta = (len(files) - (i + 1)) / rate / 60
        print(f'  {i+1:,}/{len(files):,}  rate={rate:.0f}/s  ETA={eta:.1f} min', flush=True)

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.sort_values('date', kind='stable').reset_index(drop=True)

print(f'\nTotal rows:        {len(df):,}')
print(f'Missing date:      {no_date:,}')
print(f'Missing headline:  {no_title:,}')
print(f'Date range:        {df["date"].min()} -> {df["date"].max()}')

df.to_parquet(OUT_PARQUET, index=False, compression='snappy')
df.to_csv(OUT_CSVGZ, index=False, compression='gzip')
import os
print(f'\nWrote {OUT_PARQUET}  ({os.path.getsize(OUT_PARQUET)/1e6:.1f} MB)')
print(f'Wrote {OUT_CSVGZ}    ({os.path.getsize(OUT_CSVGZ)/1e6:.1f} MB)')
print(f'\nSample:')
print(df.head(5).to_string(index=False))

WSJ_ROOT = data/WSJ_1889-1919
Total XML files: 651,139
  10,000/651,139  rate=17/s  ETA=631.9 min
  20,000/651,139  rate=25/s  ETA=428.4 min
  30,000/651,139  rate=33/s  ETA=316.4 min
  40,000/651,139  rate=41/s  ETA=246.3 min
  50,000/651,139  rate=50/s  ETA=198.6 min
  60,000/651,139  rate=60/s  ETA=164.7 min
  70,000/651,139  rate=69/s  ETA=140.2 min
  80,000/651,139  rate=78/s  ETA=121.4 min
  90,000/651,139  rate=88/s  ETA=106.8 min
  100,000/651,139  rate=97/s  ETA=95.1 min
  110,000/651,139  rate=105/s  ETA=85.6 min
  120,000/651,139  rate=114/s  ETA=77.6 min
  130,000/651,139  rate=123/s  ETA=70.8 min
  140,000/651,139  rate=131/s  ETA=64.9 min
  150,000/651,139  rate=140/s  ETA=59.9 min
  160,000/651,139  rate=148/s  ETA=55.4 min
  170,000/651,139  rate=156/s  ETA=51.5 min
  180,000/651,139  rate=164/s  ETA=48.0 min
  190,000/651,139  rate=172/s  ETA=44.8 min
  200,000/651,139  rate=179/s  ETA=42.0 min
  210,000/651,139  rate=187/s  ETA=39.3 min
  220,000/651,139  rate=194/s  

In [3]:
# ===== Cell 2: Peek at the output before exporting =====
import pandas as pd
from pathlib import Path

OUT_DIR = Path('../ProQuest TDM Studio Samples/output_files/')
df = pd.read_parquet(OUT_DIR / 'wsj_headlines_1920_1964.parquet')

# File size check
for f in [OUT_DIR / 'wsj_headlines_1920_1964.parquet',
          OUT_DIR / 'wsj_headlines_1920_1964.csv.gz']:
    if f.exists():
        mb = f.stat().st_size / 1e6
        flag = '  ⚠ OVER 30 MB CAP' if mb > 30 else ''
        print(f'{mb:7.2f} MB  {f.name}{flag}')

print(f'\nRows: {len(df):,}')
print(f'Cols: {list(df.columns)}')
print(f'Dtypes:\n{df.dtypes}')
print(f'\nDate range:        {df["date"].min()} -> {df["date"].max()}')
print(f'Missing date:      {df["date"].isna().sum():,}')
print(f'Missing headline:  {(df["headline"] == "").sum():,}')

print('\nFirst 5:')
print(df.head(5).to_string(index=False))

print('\nLast 5:')
print(df.tail(5).to_string(index=False))

print('\nRandom 10 (spot-check across the corpus):')
print(df.sample(10, random_state=0).to_string(index=False))

# Headline length sanity check — surfaces regex bugs like a runaway .*?
lens = df['headline'].str.len()
print(f'\nHeadline length — min: {lens.min()}, '
      f'median: {int(lens.median())}, max: {lens.max()}')

longest = df.loc[lens.idxmax()]
print(f'\nLongest headline ({len(longest["headline"])} chars):')
print(f'  {longest["fname"]}  {longest["date"].date() if pd.notna(longest["date"]) else "NaT"}')
print(f'  {longest["headline"][:300]}...')

# Optional: eyeball the raw XML for a random row to confirm extraction
# WSJ_ROOT = Path('../data/WSJ_XXXX-XXXX')
# sample_fname = df.sample(1, random_state=0).iloc[0]['fname']
# print(sample_fname)
# print((WSJ_ROOT / sample_fname).read_text()[:2000])

  12.58 MB  wsj_headlines_1889_1919.parquet
   8.59 MB  wsj_headlines_1889_1919.csv.gz

Rows: 651,139
Cols: ['date', 'headline', 'fname']
Dtypes:
date        datetime64[ns]
headline            object
fname               object
dtype: object

Date range:        1889-07-08 00:00:00 -> 1919-12-31 00:00:00
Missing date:      0
Missing headline:  0

First 5:
      date                headline         fname
1889-07-08     The Bank Statement. 128470608.xml
1889-07-08    Trunk Line Receipts. 128470575.xml
1889-07-08   Article 2 -- No Title 128469645.xml
1889-07-08          THE COAL TRADE 128470471.xml
1889-07-08 The Treasury Statement. 128469667.xml

Last 5:
      date                                           headline         fname
1919-12-31        INI'NTL PAPER PRESIDENT RETURNS FROM EUROPE 129753083.xml
1919-12-31 WESTERN UNION TELEGRAPH SHOWING INCREASED EARNINGS 129742963.xml
1919-12-31                                   BANK ACCEPTANCES 129759112.xml
1919-12-31                           

In [4]:
# ===== Cell 3: Shrink under the 30 MB cap =====
# Two tricks: (1) zstd level 22 instead of snappy — much better on text;
# (2) strip the uniform ".xml" suffix and store the ID as int64. Recoverable
# downstream as f'{article_id}.xml'.
import os, re
import pandas as pd
from pathlib import Path

OUT_DIR     = Path('../ProQuest TDM Studio Samples/output_files/')
SRC         = OUT_DIR / 'wsj_headlines_1920_1964.parquet'
OUT_SHRUNK  = OUT_DIR / 'wsj_headlines_1920_1964_zstd.parquet'

df = pd.read_parquet(SRC)
print(f'Loaded {len(df):,} rows from {SRC.name}')

# Strip ".xml" → integer ID. Anything that doesn't match stays as NaN
# (then we keep the original string for those — rare).
ids = df['fname'].str.extract(r'^(\d+)\.xml$', expand=False)
n_unmatched = ids.isna().sum()
if n_unmatched == 0:
    df['article_id'] = ids.astype('int64')
    df = df[['date', 'headline', 'article_id']]
    print('All fnames matched <int>.xml; stored as int64 article_id.')
else:
    print(f'  {n_unmatched:,} fnames did not match <int>.xml — keeping fname as string for those.')
    df['article_id'] = pd.to_numeric(ids, errors='coerce').astype('Int64')
    df = df[['date', 'headline', 'fname', 'article_id']]

# zstd-22 is the strongest pyarrow supports
df.to_parquet(OUT_SHRUNK, index=False, compression='zstd', compression_level=22)

mb = os.path.getsize(OUT_SHRUNK) / 1e6
flag = '  ⚠ STILL OVER 30 MB — split by year (next cell)' if mb > 30 else '  ✓ under cap'
print(f'\n{mb:7.2f} MB  {OUT_SHRUNK.name}{flag}')

# Verify roundtrip
chk = pd.read_parquet(OUT_SHRUNK)
print(f'\nRoundtrip: {len(chk):,} rows, cols={list(chk.columns)}')
print(chk.head(3).to_string(index=False))

Loaded 651,139 rows from wsj_headlines_1889_1919.parquet
All fnames matched <int>.xml; stored as int64 article_id.

   6.06 MB  wsj_headlines_1889_1919_zstd.parquet  ✓ under cap

Roundtrip: 651,139 rows, cols=['date', 'headline', 'article_id']
      date              headline  article_id
1889-07-08   The Bank Statement.   128470608
1889-07-08  Trunk Line Receipts.   128470575
1889-07-08 Article 2 -- No Title   128469645


In [5]:
# ===== Cell 4: Peek at the shrunk file =====
import os
import pandas as pd
from pathlib import Path

OUT_DIR = Path('../ProQuest TDM Studio Samples/output_files/')
SHRUNK  = OUT_DIR / 'wsj_headlines_1920_1964_zstd.parquet'

mb = os.path.getsize(SHRUNK) / 1e6
flag = '  ⚠ OVER 30 MB CAP — run the split cell next' if mb > 30 else '  ✓ under cap'
print(f'{mb:7.2f} MB  {SHRUNK.name}{flag}\n')

df = pd.read_parquet(SHRUNK)
print(f'Rows: {len(df):,}')
print(f'Cols: {list(df.columns)}')
print(f'Dtypes:\n{df.dtypes}')
print(f'\nDate range:        {df["date"].min()} -> {df["date"].max()}')
print(f'Missing date:      {df["date"].isna().sum():,}')
print(f'Missing headline:  {(df["headline"] == "").sum():,}')

print('\nFirst 5:')
print(df.head(5).to_string(index=False))

print('\nLast 5:')
print(df.tail(5).to_string(index=False))

print('\nRandom 10 (spot-check across the corpus):')
print(df.sample(10, random_state=0).to_string(index=False))

lens = df['headline'].str.len()
print(f'\nHeadline length — min: {lens.min()}, '
      f'median: {int(lens.median())}, max: {lens.max()}')

   6.06 MB  wsj_headlines_1889_1919_zstd.parquet  ✓ under cap

Rows: 651,139
Cols: ['date', 'headline', 'article_id']
Dtypes:
date          datetime64[ns]
headline              object
article_id             int64
dtype: object

Date range:        1889-07-08 00:00:00 -> 1919-12-31 00:00:00
Missing date:      0
Missing headline:  0

First 5:
      date                headline  article_id
1889-07-08     The Bank Statement.   128470608
1889-07-08    Trunk Line Receipts.   128470575
1889-07-08   Article 2 -- No Title   128469645
1889-07-08          THE COAL TRADE   128470471
1889-07-08 The Treasury Statement.   128469667

Last 5:
      date                                           headline  article_id
1919-12-31        INI'NTL PAPER PRESIDENT RETURNS FROM EUROPE   129753083
1919-12-31 WESTERN UNION TELEGRAPH SHOWING INCREASED EARNINGS   129742963
1919-12-31                                   BANK ACCEPTANCES   129759112
1919-12-31                              New York Sub-Treasury   1297590

In [6]:
# ===== Cell 5: (fallback) Split into chunks ≤ 28 MB each =====
# Only run if Cell 4's peek showed ⚠ OVER 30 MB. Greedily packs years into
# chunks while keeping each parquet under TARGET_MB. The TDM cap is per
# week, so you'll ship one chunk per week.
import os
import pandas as pd
from pathlib import Path

OUT_DIR    = Path('../ProQuest TDM Studio Samples/output_files/')
SRC        = OUT_DIR / 'wsj_headlines_1920_1964_zstd.parquet'
TARGET_MB  = 28.0   # leave headroom under the 30 MB cap

df = pd.read_parquet(SRC).sort_values('date').reset_index(drop=True)
df['year'] = df['date'].dt.year

# Estimate bytes per row from the full file, then chunk years greedily.
total_mb = os.path.getsize(SRC) / 1e6
mb_per_row = total_mb / len(df)
year_sizes = df.groupby('year').size() * mb_per_row  # approx MB per year

chunks, current, current_mb = [], [], 0.0
for y, mb in year_sizes.items():
    if current and current_mb + mb > TARGET_MB:
        chunks.append(current); current, current_mb = [], 0.0
    current.append(int(y)); current_mb += mb
if current:
    chunks.append(current)

print(f'Will write {len(chunks)} chunks (target ≤ {TARGET_MB} MB each):')
for c in chunks:
    print(f'  {c[0]}-{c[-1]}  ({len(c)} years)')

written = []
for c in chunks:
    sub = df[df['year'].between(c[0], c[-1])].drop(columns='year')
    out = OUT_DIR / f'wsj_headlines_{c[0]}_{c[-1]}.parquet'
    sub.to_parquet(out, index=False, compression='zstd', compression_level=22)
    mb = os.path.getsize(out) / 1e6
    flag = '  ⚠ OVER' if mb > 30 else ''
    print(f'  {mb:6.2f} MB  {out.name}  ({len(sub):,} rows){flag}')
    written.append(out)

print(f'\nShip one per week with the export cell below — update `data_to_export` each time.')

Will write 1 chunks (target ≤ 28.0 MB each):
  1889-1919  (31 years)
    6.06 MB  wsj_headlines_1889_1919.parquet  (651,139 rows)

Ship one per week with the export cell below — update `data_to_export` each time.


In [7]:
# ===== Cell 6: Ship to S3 results bucket =====
# If Cell 3 got you under the cap, send the zstd parquet:
data_to_export = '../ProQuest TDM Studio Samples/output_files/wsj_headlines_1920_1964_zstd.parquet'
# Otherwise, set this to one of the wsj_headlines_<startyear>_<endyear>.parquet
# files from Cell 5 and re-run this cell once per week (30 MB cap is weekly).
!aws s3 cp "$data_to_export" s3://pq-tdm-studio-results/tdm-ale-data/a4992/results/

upload: ../ProQuest TDM Studio Samples/output_files/wsj_headlines_1889_1919_zstd.parquet to s3://pq-tdm-studio-results/tdm-ale-data/a4992/results/wsj_headlines_1889_1919_zstd.parquet
